# 06 — Toyota Smarthome preflight (ONLINE, CPU, internet ON)

| attach as input | produces |
|---|---|
| `behaviorsense-code`, `behaviorsense-weights` | `behaviorsense-toyota-meta` (a few MB) |
| all nine Toyota / MSMT mounts | a go/no-go verdict, printed |

**Run this before the extraction session, not instead of it.** Extraction is a ~6 h offline
job whose output is only as good as its labels, and there are four things about this corpus
that can be wrong in ways that do not crash:

1. **The archive ships per-video CSVs, not the aggregated JSON.** `Annotation_v1.0.tar.gz`
   gives `Annotation/P18/P18T13C07.csv`. Every published baseline trains against
   `smarthome_CS_51.json`, which is a DERIVED artefact mirrored on GitHub. Both are read
   here and compared span-for-span.
2. **The CSVs carry no duration.** `frame_multilabel` maps feature step to annotation
   position as `step * duration / n_steps`, so a wrong duration rescales the whole label
   tensor and leaves its shape correct. Durations come from the JSON and are checked.
3. **Frames, not seconds.** Untrimmed is 25 fps at x1.25 speed, trimmed is 30 fps. One
   shared constant silently stretches one half by 20%.
4. **Two official id spaces for the trimmed classes**, differing by one, with 0 meaning
   "unmatched name" in the RGB one.

This notebook is ONLINE for one reason: the annotation JSON has to come off GitHub. It then
writes it to `/kaggle/working` so the OFFLINE extraction notebook can read it from a private
dataset instead of the network.

**The JSON is not committed to the repo.** `github.com/PARTHG0106/BehaviorSense` is public
and this dataset is licensed for academic research only, granted per-request. Staging it into
a private Kaggle dataset is the same pattern the other restricted assets already use.

In [ ]:
# Resolve the repo mount by CONTENT, not by dataset name.
#
# The dataset title is free text and this project has already been uploaded under more
# than one spelling ("behaviorsense-*" and "behavioursense-*"). Hard-coding the name makes
# cell 1 of a 12-hour session fail on a typo, so find the repo by a file only it contains.
import pathlib
from itertools import islice

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    # islice, NOT sorted(...)[:6]. `sorted()` materialises the whole listing
                    # first, and on Kaggle's FUSE mount a slug whose files sit at its root -
                    # `toyota-smarthome-skeleton-v1-2` holds 16,115 - makes that a full
                    # network directory read per mount. Eleven mounts of that shape is most
                    # of the 14 minutes this cell took on the first Toyota run. Six names
                    # are all this diagnostic needs, so stop after six.
                    top = sorted(islice((q.name for q in ds.iterdir()), 6)) \
                        if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

# MOUNT-INDEXED SEARCH, and why two earlier fixes were not enough.
#
# `INPUT.glob("**/x")` walks every directory under /kaggle/input - 93 minutes with the
# Toyota corpus mounted, notebook 06 measured, and 349 s for the single
# `**/src/behaviorsense/__init__.py` probe in notebook 07's second run. The first "fix",
# FIXED-DEPTH globs like `datasets/*/*/*/rtmo-l.onnx`, was depth-bounded but not
# COST-bounded: to match at depth 3 pathlib scandirs EVERY slug child directory, including
# `toyota-smarthome-skeleton-v1-2`'s 16,115-file root and MSMT17's 65,242 crops.
#
# The mounts are KNOWN at depth 2 (`datasets/<owner>/<slug>/`), so enumerate them once and
# resolve everything else with is_file() stats - one metadata call per mount per candidate,
# never a sibling-directory listing.
_SLUGS = (sorted((INPUT / "datasets").glob("*/*"))
          + sorted((INPUT / "competitions").glob("*")))

def find_fast(tail, what, required=True):
    # Known staging prefixes, each costing one stat per mount:
    #   ""                      files at the slug root
    #   EmotionSense-Extended/  the code dataset was created by zipping the repo FOLDER,
    #                           so everything sits one level below the slug
    #   kaggle/working/         Save Version nests the working directory
    # The prefixes apply to MULTI-COMPONENT tails too. They used to be tried only for bare
    # filenames, which quietly sent `src/behaviorsense/__init__.py` - the one probe every
    # notebook makes - down the deep-search path it was written to avoid.
    # `weights/` is additionally tried for a bare filename, the staged weights layout.
    prefixes = ("", "EmotionSense-Extended/", "kaggle/working/")
    mids = ("",) if "/" in tail else ("", "weights/")
    for t in [p + m + tail for p in prefixes for m in mids]:
        hits = [s / t for s in _SLUGS if (s / t).is_file()]
        if hits:
            return hits[0]
    hits = sorted(INPUT.glob(f"**/{tail}"))   # last resort: unusual layout, slow, once
    if hits:
        print(f"  {what:<9} found only by deep search ({tail}) - layout is unusual")
        return hits[0]
    if required:
        raise AssertionError(
            f"{what}: nothing matches {tail!r} in any mount. Attached: {ATTACHED}")
    print(f"  {what:<9} ABSENT (optional)")
    return None

def find_asset(pattern, what, required=True):
    # Callers pass a `**/...` pattern. The leading `**/` is stripped and the mount-indexed
    # search runs first, so every existing call site gets the speed-up unchanged.
    tail = pattern[3:] if pattern.startswith("**/") else pattern
    return find_fast(tail, what, required=required)

def find_dir(subdir, pattern, roots=None):
    # "Which mount holds the most files matching this pattern in this subdirectory?" - one
    # scandir of ONE named directory per mount, never a recursive walk. Used for corpora
    # (Toyota's mp4/ and Videos_mp4/) where the answer is a directory, not a file.
    from fnmatch import fnmatch
    import os
    best, best_n = None, 0
    for root in (roots if roots is not None else _SLUGS):
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = sum(1 for e in os.scandir(base) if e.is_file() and fnmatch(e.name, pattern))
        if n > best_n:
            best, best_n = base, n
    return best, best_n

SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"
CONFIGS = CODE / "configs"
import sys
sys.path.insert(0, str(SRC)); sys.path.insert(0, str(SCRIPTS))
print(f"  code {CODE}")
print(f"  attached {ATTACHED}")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_adl.py", "--tau-train",
     "notebook 03 passes --sampler/--tau-train; a snapshot predating them exits 2 from "
     "argparse, which notebook 03 reports as a failed stream rather than stale code"),
    ("src/behaviorsense/data/skeleton_dataset.py", "def load_subject_map",
     "video-id -> Charades actor-id remap for a person-disjoint P1 split (notebooks "
     "03/04). A stale snapshot silently reverts P1 to video-disjoint - same person in "
     "train and val - while printing numbers that look identical"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/ensemble.py", "def flip_windows",
     "test-time flip augmentation (notebook 04 levers cell). A stale snapshot would accept "
     "`clf.tta = True` as a new attribute and silently do no TTA, reporting the "
     "unaugmented number as if it were augmented"),
    ("src/behaviorsense/models/stgcnpp.py", "parents.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise. The "
     "token tracked the local name `parent` and broke when the function grew an explicit "
     "`parents` argument for Toyota's 15-node tree - a rename silently disarming a staleness "
     "guard is exactly what this list exists to catch, so it caught itself"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def force_greedy",
     "pins BOTH decoding arms to greedy (notebook 04). Without it the constrained arm "
     "inherits Qwen's generation_config (do_sample=True, temperature=0.7) while the free "
     "arm is greedy, so the comparison measures temperature instead of grammar - the "
     "constrained rate moved 8.2% -> 10.3% between two runs of identical code"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def _chat_text",
     "both arms send the SAME templated text (notebook 04). The constrained path used to "
     "hand outlines the raw prompt, so one arm got a Qwen chat turn and the other a naked "
     "instruction block - a second confound on top of the sampling one"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/eval/activity_eval.py", "def logit_adjust",
     "notebook 04's P1 cell imports MIN_SUPPORT and scores() from here, so a stale "
     "snapshot fails with ImportError at cell 3; also carries the post-hoc accuracy "
     "levers scripts/rescore_p1.py replays off the saved val logits"),
    ("src/behaviorsense/video.py", "def child_env",
     "notebook 05's /video endpoint decodes uploads in a CHILD process (ffmpeg raises SIGSEGV "
     "on malformed streams and a signal is not catchable, so without the boundary one bad "
     "upload kills the kernel, the tunnel and the demo together). `child_env` is what puts "
     "behaviorsense on that child's PYTHONPATH - sys.path does not cross a process boundary, "
     "and a snapshot without it 422s EVERY upload with \"No module named 'behaviorsense'\""),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
# Resolve the nine mounts BY CONTENT, at BOUNDED DEPTH.
#
# Never by dataset name - that rule has paid for itself twice. But the first run of this
# notebook proved the rule is not free: nine `**` globs over ~130,000 mounted files took
# **78 minutes**, because `**` descends into `mp4/` (16,115 entries) and
# `bounding_box_train/` (65,242) once per pattern.
#
# Datasets mount at `datasets/<owner>/<slug>/...`, so each asset is described by the
# SUBDIRECTORY it lives in and the pattern inside it. Testing `is_dir()` on a candidate is
# O(1), and counting is then one directory read instead of a tree walk. The observed layouts
# are all covered, including `toyota-smarthome-skeleton-v1-2` whose files sit at the slug
# root with no subfolder at all.
import collections, json, os, re, sys, time

ROOTS = sorted((INPUT / "datasets").glob("*/*")) if (INPUT / "datasets").is_dir() else []
print(f"{len(ROOTS)} dataset mount(s):")
for r in ROOTS:
    print(f"  {r.name}")

def count_dir(d, pattern):
    # scandir, not glob: for a 16k-entry directory this is one syscall loop and no Path
    # object per miss. The count is what tells a truncated upload from a complete one.
    n = 0
    try:
        with os.scandir(d) as it:
            for e in it:
                if e.is_file() and _fnmatch(e.name, pattern):
                    n += 1
    except OSError:
        return 0
    return n

from fnmatch import fnmatch as _fnmatch

# (subdir inside the slug, filename glob). "" means the slug root.
ASSETS = {
    "annotation_csv":   ("Annotation",  "P*T*C*.csv",  True),
    "rgb_untrimmed":    ("Videos_mp4",  "P*T*C*.mp4",  True),
    "pose_untrimmed":   ("Skeleton",    "results_P*T*C*_lcrnet*.json", False),
    "depth_untrimmed":  ("Depth",       "P*T*C*.mp4",  False),
    "rgb_trimmed":      ("mp4",         "*_p[0-9][0-9]_r*_c[0-9][0-9].mp4", True),
    "skel_trimmed_v11": ("json",        "*_p[0-9][0-9]_r*_c[0-9][0-9].json", False),
    "skel_trimmed_v12": ("",            "*_p[0-9][0-9]_r*_c[0-9][0-9]_pose3d.json", False),
    "depth_trimmed":    ("depth",       "*_p[0-9][0-9]_r*_c[0-9][0-9].mp4", False),
}

RESOLVED, COUNTS = {}, {}
t0 = time.time()
for key, (subdir, pattern, required) in ASSETS.items():
    best, best_n = None, 0
    for root in ROOTS:
        # The annotation CSVs are one level deeper (Annotation/P02/*.csv), so try both the
        # subdir itself and its immediate children before giving up on this root.
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = count_dir(base, pattern)
        if n == 0:
            n = sum(count_dir(c, pattern) for c in sorted(base.iterdir()) if c.is_dir())
        if n > best_n:
            best, best_n = base, n
    RESOLVED[key], COUNTS[key] = best, best_n
    if best is None or best_n == 0:
        if required:
            raise AssertionError(
                f"{key}: no mount holds {subdir or '<root>'}/{pattern}. Mounts: "
                f"{[r.name for r in ROOTS]}")
        print(f"  {key:<20} ABSENT (optional)")
    else:
        print(f"  {key:<20} {best_n:>7,} files   {best}")
print(f"resolved in {time.time() - t0:.1f}s (the `**` version took 78 minutes)")

# Expected magnitudes, from the dataset's own README. A truncated upload trains fine and
# scores slightly worse, which is the most expensive way to be wrong.
EXPECT = {"annotation_csv": 536, "rgb_untrimmed": 536, "rgb_trimmed": 16115}
print()
for key, want in EXPECT.items():
    got = COUNTS.get(key, 0)
    print(f"  {key:<20} {got:>7,} / {want:,}  "
          f"{'ok' if got == want else 'SHORT' if got < want else 'MORE THAN EXPECTED'}")

In [ ]:
# The annotation JSON, off GitHub. This is the ONE thing this notebook needs internet for.
#
# Two files: CS is the protocol every baseline reports (PDAN 32.7% f-mAP), CV is the
# cross-view one. They are read for `duration` and as an independent copy of the same spans.
import urllib.request, pathlib

BASE = ("https://raw.githubusercontent.com/dairui01/Toyota_Smarthome/main/pipline/data/")
OUT = pathlib.Path("/kaggle/working/toyota_meta")
OUT.mkdir(parents=True, exist_ok=True)

JSONS = {}
for name in ("smarthome_CS_51.json", "smarthome_CV_51.json", "Action_list"):
    dst = OUT / name
    if not dst.exists():
        urllib.request.urlretrieve(BASE + name, dst)
    print(f"  {name:<24} {dst.stat().st_size:>9,} bytes")
    if name.endswith(".json"):
        JSONS[name] = json.loads(dst.read_text(encoding="utf-8"))

# The class list is the id space. Verify the download against the tuple we train on rather
# than trusting either - a reordered Action_list would relabel the entire corpus.
from behaviorsense.data.toyota import TSU_CLASSES
_lines = [l.split("\t") for l in (OUT / "Action_list").read_text().splitlines() if l.strip()]
_names = tuple(p[1] for p in _lines)
assert _names == TSU_CLASSES, (
    "the downloaded Action_list disagrees with TSU_CLASSES. First difference at "
    f"{next(i for i,(a,b) in enumerate(zip(_names, TSU_CLASSES)) if a != b)}")
print(f"  Action_list agrees with TSU_CLASSES on all {len(TSU_CLASSES)} names")

In [ ]:
# SURVEY the label vocabulary before parsing anything for real.
#
# This cell exists because of how the first run failed. `read_annotation_csv` raised on the
# first unrecognised class name - 93 minutes in, having examined ONE file of 536 - and the
# name was `Make_coffee.Get_water`. The dataset README had the answer all along:
#
#   "please merge `Make_coffee.Get_water` & `Get Water` and `Insert_tea_bag` &
#    `Make_tea.Insert_tea_bag` to have 51 action classes"
#
# The CSVs ship the PRE-MERGE 53-name vocabulary; `Action_list` is the post-merge 51 that
# every published number is computed over. `TSU_ALIASES` now applies the merge, and the
# survey below reports EVERY unrecognised name in one pass, so a second surprise costs one
# run rather than one run per name.
from behaviorsense.data.toyota import survey_annotation_csvs, TSU_ALIASES, TSU_CLASSES

ANN_ROOT = RESOLVED["annotation_csv"]
if ANN_ROOT.name.startswith("P") and ANN_ROOT.parent.name == "Annotation":
    ANN_ROOT = ANN_ROOT.parent          # resolver may land on a per-subject subfolder

SURVEY = survey_annotation_csvs(ANN_ROOT)
print(f"{SURVEY['n_files']} CSVs, {SURVEY['n_rows']:,} annotation rows, "
      f"{SURVEY['n_distinct_names']} distinct class names")
print(f"\nmerges applied ({len(TSU_ALIASES)} aliases known):")
for raw, n in sorted(SURVEY["aliased"].items(), key=lambda kv: -kv[1]):
    print(f"  {raw:<28} -> {TSU_ALIASES[raw]:<20} {n:>6,} rows")

if SURVEY["unknown"]:
    print(f"\nSTOP: {len(SURVEY['unknown'])} class name(s) resolve to no class:")
    for name, where in SURVEY["unknown"].items():
        print(f"  {name!r}  first seen in {where}")
    print("\nAdd them to TSU_ALIASES (if they are merges) or to TSU_CLASSES (if the")
    print("Action_list is incomplete), re-upload behaviorsense-code, and re-run.")
    raise SystemExit("unknown class names - see above")
print(f"\nGO: every name resolves into the {len(TSU_CLASSES)}-class space.")

In [ ]:
# Parse the CSVs, then check them against the JSON span-for-span. This is the gate.
from behaviorsense.data.toyota import (TSU_FPS, cross_check_annotations, frame_counts,
                                       load_tsu_annotations, load_tsu_annotations_from_csv,
                                       read_annotation_csv, collapse_matrix, COARSE_V11,
                                       TSU_TO_COARSE, iter_videos, PROTOCOLS)

JSON_SIDE = load_tsu_annotations(OUT / "smarthome_CS_51.json")
print(f"JSON: {len(JSON_SIDE)} videos, official CS partition verified on read")

# Show one raw CSV before parsing anything, so the confirmed schema is visible and not
# taken on faith. The first run printed exactly this and it is how the merge was found.
_one = sorted(ANN_ROOT.glob("**/P*T*C*.csv"))[0]
print(f"\nraw {_one.name}:")
for _l in _one.read_text(encoding="utf-8-sig").splitlines()[:5]:
    print(f"    {_l}")
print(f"parsed -> {read_annotation_csv(_one)[:3]}")

DURATIONS = {v: JSON_SIDE[v].duration for v in JSON_SIDE}
CSV_SIDE = load_tsu_annotations_from_csv(ANN_ROOT, DURATIONS)
print(f"\nCSV:  {len(CSV_SIDE)} videos parsed")

REPORT = cross_check_annotations(CSV_SIDE, JSON_SIDE)
print(f"\ncross-check: {REPORT['compared']} videos compared, "
      f"{REPORT['agree']} span-identical, {len(REPORT['disagree'])} span-different")
print(f"FRAME-level agreement: mean {REPORT['frame_agreement_mean']:.5f}  "
      f"worst {REPORT['frame_agreement_min']:.5f}  "
      f"videos below 0.999: {REPORT['videos_below_999']}")
print()
print("Span equality is the wrong yardstick on its own and the first run showed why: the")
print("CSVs carry FINER spans and the JSON merges adjacent same-class intervals, so")
print("(2,1340,2960)+(2,2960,2963) against (2,1480,2963) counts as a disagreement while")
print("describing almost the same frames. Frames are what the model sees, so that is what")
print("is scored. A pure split scores 1.00000; only a moved boundary costs anything.")
print()
for d in REPORT["disagree"][:5]:
    print(f"  {d['vid']}: csv {d['csv_spans']} spans vs json {d['json_spans']}, "
          f"frame agreement {d['frame_agreement']:.5f} "
          f"({d['cells_differing']:,} cells differ)")
if REPORT["csv_only_videos"]:
    print(f"  only in CSVs ({len(REPORT['csv_only_videos'])}): "
          f"{REPORT['csv_only_videos'][:5]}")
if REPORT["json_only_videos"]:
    print(f"  only in JSON ({len(REPORT['json_only_videos'])}): "
          f"{REPORT['json_only_videos'][:5]}")
print()
if REPORT["frame_agreement_mean"] >= 0.999 and REPORT["compared"] == len(JSON_SIDE):
    print("GO. The two sources describe the same frames to better than 1 part in 1,000.")
    print("Train on the JSON, because that is what every published baseline reports on")
    print("(PDAN 32.7% f-mAP, CS) - the CSVs are authoritative but not comparable.")
elif REPORT["frame_agreement_mean"] >= 0.99:
    print("GO WITH A CAVEAT. Frame agreement is 0.99-0.999, so a minority of videos have")
    print("genuinely moved boundaries rather than merged spans. Train on the JSON for")
    print("comparability and record the agreement figure next to the numbers.")
else:
    print("STOP. Frame agreement below 0.99 means the sources disagree about content, not")
    print("just about how spans are grouped. Do not spend an extraction session on either")
    print("until the videos listed above have been looked at.")

In [ ]:
# The distribution, printed so the extraction pass can be checked against it later.
import numpy as np

FINE = frame_counts(JSON_SIDE.values())
M = collapse_matrix(TSU_CLASSES, TSU_TO_COARSE)
COARSE = FINE @ M
TOT = float(FINE.sum())
print(f"{TOT:,.0f} annotated frames = {TOT / TSU_FPS / 3600:.1f} h at {TSU_FPS:g} Hz")
for p in PROTOCOLS:
    sides = {s: len(list(iter_videos(JSON_SIDE, p, s)))
             for s in ("train", "val", "test", "unused")}
    print(f"  {p:<4} " + "  ".join(f"{k} {v}" for k, v in sides.items() if v))

print(f"\n{'coarse class':<24}{'frames':>12}{'share%':>8}")
for i in np.argsort(-COARSE):
    if COARSE[i]:
        print(f"{COARSE_V11[i]:<24}{int(COARSE[i]):>12,}{100 * COARSE[i] / TOT:>7.2f}")
_idle = int(COARSE[COARSE_V11.index("other_idle")])
print(f"\nother_idle: {_idle} frames. Zero is the intended answer - every Toyota class is a "
      "real activity, so nothing needs the reject class.")

_gaps = np.array([1 - v.annotated_frames() / max(1, v.duration) for v in JSON_SIDE.values()])
print(f"unannotated: mean {_gaps.mean():.3f} median {np.median(_gaps):.3f} max {_gaps.max():.3f}")
print("A third of the corpus is gap. The 51-class benchmark head treats gaps as true")
print("negatives; the unified head must MASK them, because a gap most likely contains the")
print("sitting and standing that Toyota never labels.")

In [ ]:
# Confirm the RGB actually decodes, and that the filename parsers survive the real names.
# One video per protocol side, because a corrupt archive member is cheaper to find now than
# 4 hours into extraction.
import cv2
from behaviorsense.data.toyota import (TSM_FPS, TSM_FPS_DOCUMENTED, parse_tsm_name,
                                       parse_tsu_filename,
                                       protocol_side)

def probe(path):
    cap = cv2.VideoCapture(str(path))
    ok = cap.isOpened()
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    got, _frame = cap.read()
    cap.release()
    return ok and got, n, fps, w, h

print("untrimmed RGB (expect 25 fps, 640x480):")
for side in ("train", "test"):
    # islice, not a full sorted glob: 536 Path objects is cheap but 16,115 is not, and the
    # trimmed loop below would pay it for two probes.
    import itertools
    for p in itertools.islice(RESOLVED["rgb_untrimmed"].glob("P*T*C*.mp4"), 200):
        v = parse_tsu_filename(p)
        if protocol_side(v, "CS") != side:
            continue
        ok, n, fps, w, h = probe(p)
        d = JSON_SIDE.get(v.tsu)
        drift = "" if d is None else f"  json duration {d.duration} (delta {n - d.duration:+d})"
        print(f"  [{side}] {p.name}: decode={ok} frames={n:,} {fps:g}fps {w}x{h}{drift}")
        break

print(f"\ntrimmed RGB (expect {TSM_FPS:g} fps from the container, {TSM_FPS_DOCUMENTED:g} in the README):")
for p in itertools.islice(RESOLVED["rgb_trimmed"].glob("*.mp4"), 2):
    v = parse_tsm_name(p.name)
    ok, n, fps, w, h = probe(p)
    verdict = ("matches the container constant" if abs(fps - TSM_FPS) < 0.6 else
               "matches the README" if abs(fps - TSM_FPS_DOCUMENTED) < 0.6 else
               f"matches NEITHER - {fps:g} is a third value")
    print(f"  {p.name}: {v.activity} p{v.subject:02d} c{v.camera:02d} "
          f"decode={ok} frames={n} {fps:g}fps {w}x{h}  -> {verdict}")
    print(f"      {n} frames = {n / max(fps, 1):.1f}s at the container rate, "
          f"{n / TSM_FPS_DOCUMENTED:.1f}s at the README's {TSM_FPS_DOCUMENTED:g}")
print("  The README and the files disagree, so the extractor reads fps PER FILE and this")
print("  cell is what says which value to expect. Assuming either one silently rescales")
print("  every trimmed duration by 1.5x.")

if RESOLVED["skel_trimmed_v12"] is not None:
    _s = next(iter(RESOLVED["skel_trimmed_v12"].glob("*_pose3d.json")))
    _v = parse_tsm_name(_s.name)
    print(f"\nV1.2 skeleton: {_s.name} -> {_v.activity} p{_v.subject:02d} tag={_v.tag!r}")
    print("  (the tag group is why this archive parses at all - an anchored pattern without")
    print("   it rejects every file here and reports it as an empty mount)")

In [ ]:
# Stage the metadata for the OFFLINE extraction notebook, plus the resolved manifest.
#
# The manifest records which mount answered which pattern and how many files it held. When
# extraction later disagrees with these numbers, the question 'did the mounts change?' has a
# recorded answer instead of a reconstruction.
MANIFEST = {
    "resolved": {k: (str(v) if v else None) for k, v in RESOLVED.items()},
    "counts": COUNTS,
    "expected": EXPECT,
    "cross_check": {k: v for k, v in REPORT.items() if k != "disagree"},
    "n_disagree": len(REPORT["disagree"]),
    "fine_frame_counts": FINE.tolist(),
    "tsu_fps": TSU_FPS, "tsm_fps": TSM_FPS,
    "durations": DURATIONS,
}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=1), encoding="utf-8")
print(f"wrote {OUT}/ :")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<26} {f.stat().st_size:>10,} bytes")

print()
print("=" * 70)
print("  Save Version -> create a PRIVATE dataset named behaviorsense-toyota-meta")
print("=" * 70)
print("The offline extraction notebook attaches that instead of reaching for GitHub.")
print()
print("Do not commit these files to the repo: it is PUBLIC and this dataset is licensed")
print("for academic research only. .gitignore already refuses them.")